# GPU-Accelerated Sparse Linear Algebra Network Intrusion Detection System
### Reproducible NVIDIA GPU (CUDA / cuSPARSE) Execution & Benchmark on Google Colab

This notebook configures, builds, and evaluates the **GPU Sparse IDS** engine.

**Hardware Requirement:**
Ensure you are running on a GPU runtime: `Runtime` -> `Change runtime type` -> select `T4 GPU` (or V100/A100).

**Core Motif:** Sparse linear algebra ($C = P \times S^T$) with byte n-gram feature representation + exact substring verification.

## Step 1: Hardware & Environment Detection
Check for NVIDIA GPU availability, driver status, and CUDA toolkit version.

In [ ]:
import subprocess, sys, os

print("=== Detecting NVIDIA GPU ===")
try:
    smi_out = subprocess.check_output(["nvidia-smi"], text=True)
    print(smi_out)
    has_gpu = True
except Exception as e:
    print("WARNING: nvidia-smi failed or no NVIDIA GPU detected:", e)
    has_gpu = False

try:
    nvcc_out = subprocess.check_output(["nvcc", "--version"], text=True)
    print("=== CUDA Compiler (nvcc) ===")
    print(nvcc_out)
except Exception as e:
    print("WARNING: nvcc not found in PATH:", e)

if not has_gpu:
    print("\n[FALLBACK NOTICE] No NVIDIA GPU found in this Colab session.")
    print("Please navigate to Runtime -> Change runtime type -> Hardware accelerator -> T4 GPU.")
    print("The notebook can still run in CPU (Sequential / OpenMP) fallback mode.")

## Step 2: Clone or Accept Repository
Clone from GitHub or use the uploaded workspace.

In [ ]:
import os

REPO_DIR = "/content/gpu_sparse_ids"
REPO_URL = "https://github.com/Nikz1476/gpu-sparse-ids.git"

if not os.path.exists(REPO_DIR):
    print(f"Cloning {REPO_URL} into {REPO_DIR}...")
    !git clone {REPO_URL} {REPO_DIR}
else:
    print(f"Updating existing repository at {REPO_DIR}...")
    !cd {REPO_DIR} && git fetch origin && git reset --hard origin/main

%cd {REPO_DIR}
!git log -n 3 --oneline


## Step 3: Install Required Dependencies
Install `libpcap-dev`, `cmake`, `build-essential`, and python plotting libraries.

In [ ]:
!apt-get update -qq
!apt-get install -y -qq build-essential cmake libpcap-dev git
!pip install -q matplotlib numpy

## Step 4: Configure & Build CUDA / cuSPARSE Engine
Auto-detect the GPU compute architecture (T4=75, V100=70, A100=80) and compile.

In [ ]:
import torch, os

cuda_arch = "75"  # Default for Colab T4
if torch.cuda.is_available():
    major, minor = torch.cuda.get_device_capability()
    cuda_arch = f"{major}{minor}"
    gpu_name = torch.cuda.get_device_name(0)
    print(f"Detected GPU: {gpu_name} (Compute capability: sm_{cuda_arch})")
else:
    print("GPU not detected via PyTorch, defaulting sm_75")

# Configure CMake with CUDA enabled
!mkdir -p build_cuda
%cd build_cuda
!cmake .. -DCMAKE_BUILD_TYPE=Release -DUSE_CUDA=ON -DCMAKE_CUDA_ARCHITECTURES={cuda_arch}
!make -j$(nproc)
%cd ..

## Step 5: Generate Test Data & Benchmarks
Produce ground-truth synthetic PCAP and attack signatures.

In [ ]:
!mkdir -p data/pcap data/signatures benchmarks output
!./build_cuda/generate_test_data data 10000
!ls -la data/pcap data/signatures

## Step 6: Run Correctness Test Suite
Validate all 23 unit, property, and integration tests.

In [ ]:
!./build_cuda/test_ids

## Step 7: Run CUDA IDS on Ground-Truth Traffic
Execute the full pipeline with cuSPARSE SpGEMM, exporting structured CSV & JSON.

In [ ]:
!./build_cuda/sparse_ids data/pcap/test_traffic.pcap data/signatures/test_signatures.txt cuda 1 3 65536 \
    --out-csv output/cuda_alerts.csv \
    --out-json output/cuda_alerts.json

import json
if os.path.exists("output/cuda_alerts.json"):
    with open("output/cuda_alerts.json") as f:
        res = json.load(f)
    print(f"Generated {len(res['alerts'])} verified alerts via CUDA cuSPARSE.")
    print("JSON Summary:", json.dumps(res['summary'], indent=2))

## Step 8: Comparative Benchmark (Sequential vs OpenMP vs CUDA)
Run the multi-threaded and GPU benchmarks on 10,000 packets.

In [ ]:
!./build_cuda/benchmark_ids data/pcap/benchmark_traffic.pcap data/signatures/test_signatures.txt benchmarks/benchmark_results.csv
!cat benchmarks/benchmark_results.csv

## Step 9: Alert Equivalence Validation
Confirm that Sequential CPU, OpenMP multi-thread, and CUDA GPU produce 100% identical alerts.

In [ ]:
# Run Sequential
!./build_cuda/sparse_ids data/pcap/test_traffic.pcap data/signatures/test_signatures.txt seq 1 3 65536 --out-json output/seq.json
# Run OpenMP
!./build_cuda/sparse_ids data/pcap/test_traffic.pcap data/signatures/test_signatures.txt omp 4 3 65536 --out-json output/omp.json
# Run CUDA
!./build_cuda/sparse_ids data/pcap/test_traffic.pcap data/signatures/test_signatures.txt cuda 1 3 65536 --out-json output/cuda.json

with open('output/seq.json') as f1, open('output/omp.json') as f2, open('output/cuda.json') as f3:
    s_alerts = json.load(f1)['alerts']
    o_alerts = json.load(f2)['alerts']
    c_alerts = json.load(f3)['alerts']

print(f"Sequential alerts: {len(s_alerts)}")
print(f"OpenMP alerts:     {len(o_alerts)}")
print(f"CUDA alerts:       {len(c_alerts)}")
assert len(s_alerts) == len(o_alerts) == len(c_alerts), "Alert count mismatch!"
print("SUCCESS: Sequential, OpenMP, and CUDA outputs are 100% equivalent!")

## Step 10: Generate Benchmark Plots & Roofline Analysis

In [ ]:
!python3 scripts/plot_benchmarks.py

from IPython.display import Image, display
for plot_name in ["total_time.png", "speedup.png", "phase_breakdown.png", "throughput.png"]:
    path = os.path.join("benchmarks", plot_name)
    if os.path.exists(path):
        print(f"Displaying {plot_name}:")
        display(Image(path))

## Step 11: Export & Download Benchmark Artifacts

In [ ]:
!zip -r /content/gpu_sparse_ids_results.zip benchmarks/ output/
try:
    from google.colab import files
    files.download('/content/gpu_sparse_ids_results.zip')
    print("Downloaded results bundle!")
except Exception as e:
    print("Running outside Colab interactive environment, results saved to /content/gpu_sparse_ids_results.zip")